# Chapter 6 &mdash; Pruning Unreachable States

**Concept 3 of the Chapter 6 decomposition:** *Pruning Unreachable States*

The product construction makes disconnected states; BFS from $q_0$ for $|Q|-1$ steps keeps only what matters.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Pruning-Unreachable/Concept-Pruning-Unreachable.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The product construction creates **every** pair, including pairs no input can ever
reach. Those states are harmless but useless: they clutter the drawing and slow every
later algorithm.

**Pruning** is a breadth-first search from $q_0$. Since the longest simple path has
$|Q|-1$ edges, $|Q|-1$ rounds suffice &mdash; that bound is the reason the algorithm
terminates without a visited-set argument.

Pruning is **not** minimization: it removes states you cannot get to, not states you
cannot tell apart.

## 2. Definitions

### Two machines whose product has unreachable pairs

In [ ]:
A = md2mc('''DFA
I : 0 -> F
I : 1 -> I
F : 0 -> F
F : 1 -> I
''')
B = md2mc('''DFA
I : 0 -> I
I : 1 -> F
F : 0 -> I
F : 1 -> F
''')

### Reachability by BFS, bounded by $|Q|-1$ rounds

In [ ]:
def reachable(D):
    frontier, seen = {D["q0"]}, {D["q0"]}
    for _ in range(len(D["Q"]) - 1):
        nxt = {step_dfa(D, q, c) for q in frontier for c in D["Sigma"]} - seen
        if not nxt: break
        seen |= nxt; frontier = nxt
    return seen

## 3. Tests

The raw product has states the BFS never touches.

In [ ]:
P = union_dfa(A, B)
r = reachable(P)
print("product states : %d,  reachable : %d" % (len(P["Q"]), len(r)))
print("unreachable    :", sorted(P["Q"] - r))

`pruneUnreach` removes exactly those, and the language is unchanged.

In [ ]:
Pr = pruneUnreach(P)
print("after pruneUnreach : %d states" % len(Pr["Q"]))
assert Pr["Q"] == reachable(P)
assert langeq_dfa(P, Pr)
print("same language? ", langeq_dfa(P, Pr))

Pruning is **not** minimization &mdash; they remove different things.

In [ ]:
print("raw product   : %d states" % len(P["Q"]))
print("pruned        : %d states" % len(Pr["Q"]))
print("minimized     : %d states" % len(min_dfa(P)["Q"]))
print()
print("pruning removes UNREACHABLE states;")
print("minimizing then merges INDISTINGUISHABLE ones (Concepts 6-10).")

The $|Q|-1$ bound really is enough: a longer chain still converges in time.

In [ ]:
chain = md2mc('''DFA
I  : 0 -> A
A  : 0 -> B
B  : 0 -> C
C  : 0 -> F
F  : 0 -> F
I  : 1 -> I
A  : 1 -> A
B  : 1 -> B
C  : 1 -> C
F  : 1 -> F
''')
print("|Q| =", len(chain["Q"]), " all reachable?", reachable(chain) == chain["Q"])
assert reachable(chain) == chain["Q"]

## 4. Animation

The pruned union &mdash; only the pairs an input can actually reach.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(pruneUnreach(union_dfa(A, B)), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Construct a DFA where **half** the product states are unreachable.
2. Why does BFS need only $|Q|-1$ rounds and not $|Q|$?
3. Can an unreachable state be *distinguishable* from a reachable one? Does it matter?

In [ ]:
# Your work for the exercises above.